In [83]:
import os, json
import librosa
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
)

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefixes = json.load(f)

whisper_processor = WhisperProcessor.from_pretrained(
    "models/whisper-small",
    local_files_only=True,
)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "models/whisper-small",
    dtype=torch.float16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    "models/qwen2.5-0.5b",
    local_files_only=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    "models/qwen2.5-0.5b",
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))),
    key=lambda x: int(x) if x.isdigit() else x,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [84]:
def ask_ai(transcripts, first, second):
    data = json.dumps(transcripts, ensure_ascii=False)

    prompt = f"""
Task: Reconstruct the chronological order of a shuffled English dialogue.

Each chunk has an ID and its spoken text.
The first two chunks are already known.

Return JSON with the key "order".
The value must be a list of all chunk IDs in chronological order.

Example:

Input:
first_two: [2, 0]
chunks:
[
  {{"id": 0, "text": "How can I help you?"}},
  {{"id": 1, "text": "Thank you very much."}},
  {{"id": 2, "text": "Hello!"}}
]

Output:
{{"order": [2, 0, 1]}}

Now complete the following task.

Input:
first_two: [{first}, {second}]
chunks:
{data}

Output:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(generated, skip_special_tokens=True,).strip()

In [85]:
def trans_audio(audio_path):
    au, _ = librosa.load(audio_path, sr=16000)

    features = whisper_processor(au, sampling_rate=16000, return_tensors="pt",).input_features.to(device=device, dtype=next(whisper_model.parameters()).dtype)

    with torch.inference_mode():
        ids = whisper_model.generate(features, language="en", task="transcribe", max_new_tokens=128)

    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

In [86]:
text = trans_audio("dataset/train/0/chunk_0.wav")
print(text)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I know it's going to save me a lot of money, I hope.


In [87]:
def parse_response(response, n, first, second):
    decoder = json.JSONDecoder()

    for i, char in enumerate(response):
        if char != "{":
            continue

        try:
            data, _ = decoder.raw_decode(response[i:])
        except json.JSONDecodeError:
            continue

        if not isinstance(data, dict):
            continue

        order = data.get("order")

        if not isinstance(order, list):
            continue

        if not all(type(x) is int for x in order):
            continue

        if len(order) != n:
            continue

        if sorted(order) != list(range(n)):
            continue

        if order[:2] != [first, second]:
            continue

        return order

    return None

In [88]:
import os, json
import time
from pathlib import Path

answers = {}

start_time = time.perf_counter()
qwen_success = 0

for did, (first, second) in prefixes.items():
    folder = Path(TEST_DIR) / did
    files = sorted(
        folder.glob("chunk_*.wav"),
        key=lambda p: int(p.stem.split("_")[1])
    )

    n = len(files)

    transcripts = []

    for file in files:
        chunk_id = int(file.stem.split("_")[1])

        text = trans_audio(str(file))

        transcripts.append({
            "id": chunk_id,
            "text": text
        })

    response = ask_ai(transcripts, first, second)

    sequence = parse_response(response, n, first, second)

    if sequence is None:
        print(f"Dialogue {did}: invalid Qwen response")
        print("Response:", repr(response[:200]))

        sequence = [first, second] + [
            i for i in range(n)
            if i not in (first, second)
        ]
    else:
        qwen_success += 1

    order = [0] * n
    for pos, chunk_id in enumerate(sequence):
        order[chunk_id] = pos

    answers[did] = order

    print(f"Processed dialogue {did}")

with open(OUTPUT, "w") as f:
    json.dump(answers, f)

print("Finished!")
print(f"Qwen valid responses: {qwen_success}/{len(prefixes)}")
print(f"Elapsed: {time.perf_counter() - start_time:.1f}s")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed dialogue 11


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Dialogue 69: invalid Qwen response
Response: '{"order": [15, 7, 12, 10, 9, 8, 11, 14, 13, 16, 1, 2, 3, 4, 5, 6, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51'
Processed dialogue 69


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed dialogue 202


... (30 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed dialogue 1448


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Dialogue 1494: invalid Qwen response
Response: '{"order": [6, 5, 4, 3, 2, 1]}\n\nNote:\n- The input will always be valid.\n- The output will always be valid.\n- The order of the chunks in the input is not important.\n- The order of the chunks in the outp'
Processed dialogue 1494
Finished!
Qwen valid responses: 10/20
Elapsed: 59.4s


In [89]:
from pathlib import Path

print("Current directory:", Path.cwd())

for name in [
    "wav2vec2-base-960h",
    "whisper-small",
    "qwen2.5-0.5b",
]:
    path = Path("models") / name

    print(f"\n{name}")
    print("Exists:", path.exists())

    if path.exists():
        print("Files:", [p.name for p in path.iterdir()])

Current directory: /home/kolja/home/kole4ka_bot/first-ml-project/training/ioai/2026/Find_the_Order

wav2vec2-base-960h
Exists: True
Files: ['feature_extractor_config.json', 'tokenizer_config.json', 'model.safetensors', 'vocab.json', 'config.json', 'preprocessor_config.json', '.cache', 'special_tokens_map.json']

whisper-small
Exists: True
Files: ['merges.txt', 'tokenizer_config.json', 'generation_config.json', 'normalizer.json', 'tokenizer.json', 'model.safetensors', 'added_tokens.json', 'vocab.json', 'config.json', 'preprocessor_config.json', '.cache', 'special_tokens_map.json']

qwen2.5-0.5b
Exists: True
Files: ['merges.txt', 'tokenizer_config.json', 'generation_config.json', 'tokenizer.json', 'model.safetensors', 'vocab.json', 'config.json', '.cache']


In [90]:
import json
from pathlib import Path

pred = json.loads(Path("answers.json").read_text())
truth = json.loads(
    Path("hf-data/public/pretest_answers.json").read_text()
)

def score_dialogue(p, t):
    n = len(t)

    if len(p) != n or sorted(p) != list(range(n)):
        return 0.0

    if n < 2:
        return 1.0

    correct = sum(
        (p[i] - p[j]) * (t[i] - t[j]) > 0
        for i in range(n)
        for j in range(i + 1, n)
    )

    return correct / (n * (n - 1) // 2)

scores = [
    score_dialogue(pred[k], truth[k])
    for k in truth
]

print(f"IOAI Score: {sum(scores) / len(scores):.6f}")
print(f"Dialogues evaluated: {len(scores)}")

IOAI Score: 0.688119
Dialogues evaluated: 20


In [91]:
def baseline_prediction(n, first, second):
    sequence = [first, second] + [
        i for i in range(n)
        if i not in (first, second)
    ]

    ranks = [0] * n

    for pos, chunk_id in enumerate(sequence):
        ranks[chunk_id] = pos

    return ranks


baseline_scores = []

for did, (first, second) in prefixes.items():
    n = len(truth[did])

    prediction = baseline_prediction(n, first, second)

    baseline_scores.append(
        score_dialogue(prediction, truth[did])
    )

print(f"Baseline Score: {sum(baseline_scores) / len(baseline_scores):.6f}")

Baseline Score: 0.689534


In [92]:
results = []

for did, (first, second) in prefixes.items():
    target = truth[did]

    baseline = baseline_prediction(
        len(target), first, second
    )

    qwen_score = score_dialogue(pred[did], target)
    baseline_score = score_dialogue(baseline, target)

    results.append({
        "id": did,
        "qwen": qwen_score,
        "baseline": baseline_score,
        "difference": qwen_score - baseline_score,
        "changed": pred[did] != baseline,
    })

improved = [r for r in results if r["difference"] > 1e-9]
worse = [r for r in results if r["difference"] < -1e-9]
equal = [r for r in results if abs(r["difference"]) <= 1e-9]
changed = [r for r in results if r["changed"]]

print(f"Improved: {len(improved)}")
print(f"Worse:    {len(worse)}")
print(f"Equal:    {len(equal)}")
print(f"Different predictions: {len(changed)}")

if changed:
    avg_delta = sum(r["difference"] for r in changed) / len(changed)
    print(f"Mean improvement when changed: {avg_delta:+.6f}")

print("\nPer-dialogue results:")

for r in sorted(results, key=lambda x: x["difference"]):
    print(
        f"Dialogue {r['id']:>5}: "
        f"Qwen={r['qwen']:.3f} | "
        f"Baseline={r['baseline']:.3f} | "
        f"Delta={r['difference']:+.3f}"
    )

Improved: 2
Worse:    1
Equal:    17
Different predictions: 3
Mean improvement when changed: -0.009432

Per-dialogue results:
Dialogue    11: Qwen=0.667 | Baseline=0.714 | Delta=-0.048
Dialogue    69: Qwen=0.693 | Baseline=0.693 | Delta=+0.000
Dialogue   202: Qwen=0.622 | Baseline=0.622 | Delta=+0.000
Dialogue   268: Qwen=0.806 | Baseline=0.806 | Delta=+0.000
Dialogue   340: Qwen=0.676 | Baseline=0.676 | Delta=+0.000
Dialogue   384: Qwen=0.536 | Baseline=0.536 | Delta=+0.000
Dialogue   446: Qwen=0.667 | Baseline=0.667 | Delta=+0.000
Dialogue   456: Qwen=0.810 | Baseline=0.810 | Delta=+0.000
Dialogue   570: Qwen=0.603 | Baseline=0.603 | Delta=+0.000
Dialogue   577: Qwen=0.583 | Baseline=0.583 | Delta=+0.000
Dialogue   649: Qwen=0.679 | Baseline=0.679 | Delta=+0.000
Dialogue   682: Qwen=0.610 | Baseline=0.610 | Delta=+0.000
Dialogue   697: Qwen=0.606 | Baseline=0.606 | Delta=+0.000
Dialogue  1149: Qwen=0.911 | Baseline=0.911 | Delta=+0.000
Dialogue  1257: Qwen=0.857 | Baseline=0.857 | De